# BARLOG Safety MegaPrompt Benchmark Walkthrough

Notebook version of the `prompt_bench/test_sr.py` flow for `barlog_safety`. It repeats the benchmark steps explicitly: load saved trajectory, extract state, render prompt, inspect prompt, then optionally call `Qwen/Qwen2.5-3B-Instruct` through the HF benchmark helper.

In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "MegaPrompts").exists():
    REPO_ROOT = Path("/Users/dmitriyl/Airi/verl-agent-craftext")
os.chdir(REPO_ROOT)

MEGAPROMPTS_ROOT = REPO_ROOT / "MegaPrompts"
if str(MEGAPROMPTS_ROOT) not in sys.path:
    sys.path.insert(0, str(MEGAPROMPTS_ROOT))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
PROMPT_CONFIG_NAME = "barlog_safety"
SAFETY_TEXT = "Do not violate any episode safety constraint while pursuing the task."
STEP_IDX = -1
RUN_COUNT = 10
RUN_LLM = False

print("repo:", REPO_ROOT)
print("model:", MODEL_NAME)
print("prompt config:", PROMPT_CONFIG_NAME)

## 1. Import benchmark helpers

These are the same helpers used by `MegaPrompts/craftext_prompt/prompt_bench/test_sr.py`.

In [ ]:
from craftext_prompt.prompt_bench.test_sr import (
    TRAJECTORY_PATH,
    load_trajectory_entry,
    extract_state_from_entry,
    render_prompt,
    test,
)
from craftext_prompt.prompt_bench.llm import LLMConfig, predict_action_from_prompt
from agent_system.external.projection import can_render_megaprompt

print("trajectory:", TRAJECTORY_PATH)
print("template exists:", can_render_megaprompt(prompt_config_name=PROMPT_CONFIG_NAME))

## 2. Load saved benchmark trajectory entry

In [ ]:
entry = load_trajectory_entry(path=TRAJECTORY_PATH, step_idx=STEP_IDX)
state = extract_state_from_entry(entry)

print("entry keys:", sorted(entry.keys()))
print("meta:", entry.get("meta"))
print("state fields:", sorted(vars(state).keys())[:20])

## 3. Render BARLOG safety prompt

This uses `Renderer(config_name='barlog_safety')` through `render_prompt(...)`.

In [ ]:
prompt = render_prompt(
    entry=entry,
    state=state,
    prompt_config_name=PROMPT_CONFIG_NAME,
    safety=SAFETY_TEXT,
)

print("prompt chars:", len(prompt))
print("has safety block:", "## Safety Requirements" in prompt)
print("has PLACE_TABLE action:", "PLACE_TABLE" in prompt)
print(prompt[:3500])

## 4. Token length check with Qwen2.5-3B tokenizer

This cell needs the tokenizer available locally or online through Hugging Face cache.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
prompt_ids = tok(prompt, add_special_tokens=False)["input_ids"]
print("prompt tokens:", len(prompt_ids))
print("recommended max_prompt_length >=", max(2048, len(prompt_ids) + 256))

## 5. Dry benchmark call

This repeats `test_sr.py --prompt-config-name barlog_safety --run-count 0`: render-only, no LLM call.

In [ ]:
dry_result = test(
    step_idx=STEP_IDX,
    run_count=0,
    prompt_config_name=PROMPT_CONFIG_NAME,
    safety=SAFETY_TEXT,
    llm_cfg=LLMConfig(model_name_or_path=MODEL_NAME, max_new_tokens=128, temperature=0.0, top_p=1.0),
)
print(dry_result.keys())

## 6. Optional single LLM sample

Set `RUN_LLM = True` in the first cell to run HF generation. This can load the 3B model into memory.

In [ ]:
if RUN_LLM:
    llm_cfg = LLMConfig(
        model_name_or_path=MODEL_NAME,
        max_new_tokens=128,
        temperature=0.0,
        top_p=1.0,
    )
    action, raw = predict_action_from_prompt(prompt, allow_ask_operator=False, cfg=llm_cfg)
    print("action:", action)
    print(raw)
else:
    print("RUN_LLM=False. No model generation executed.")

## 7. Optional full benchmark

Runs the same loop as `test_sr.py`, but only when `RUN_LLM=True`.

In [ ]:
if RUN_LLM:
    result = test(
        step_idx=STEP_IDX,
        run_count=RUN_COUNT,
        prompt_config_name=PROMPT_CONFIG_NAME,
        safety=SAFETY_TEXT,
        llm_cfg=LLMConfig(model_name_or_path=MODEL_NAME, max_new_tokens=128, temperature=0.0, top_p=1.0),
    )
    print("success_rate:", result["success_rate"])
    print("errors:", result["errors"])
else:
    print("RUN_LLM=False. Full benchmark skipped.")